# **The Structure of the TPCRP Algorithm:**

  1. Load CIFAR-10 (unlabeled pool)

  2. Train SimCLR (self-supervised) > extract 512-dim L2-normalised embeddings

  3. K-Means cluster the embeddings into K = min(|L| + B, 500) clusters,

     dropping clusters with fewer than 5 points

  4. Greedily select B points one at a time:

     a. Pick the cluster with the fewest labeled points (tiebreak: largest cluster)

     b. Compute intra-cluster typicality using K = min(20, cluster_size - 1) neighbours

     c. Select the unlabeled point with the highest typicality

  5. Label those B points > train a classifier > measure accuracy

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
import numpy as np
from torch.utils.data import DataLoader, Subset
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt
from tqdm import tqdm
import torchvision.models as models

## Step 1: Load CIFAR-10

We load the CIFAR-10 dataset three ways:
- **SimCLR loader**: returns two augmented views of each image for contrastive training
- **Pool loader**: returns clean images for embedding extraction after training
- **Test loader**: returns clean test images for accuracy evaluation

All images are normalised using CIFAR-10 mean and standard deviation.

In [3]:
# Augmentation for SimCLR, two views of the same image
class TwoViewTransform:
    def __init__(self, transform):
        self.transform = transform

    def __call__(self, img):
        return self.transform(img), self.transform(img)

def get_simclr_transform():
    transform = transforms.Compose([
        transforms.RandomResizedCrop(size=32),
        transforms.RandomHorizontalFlip(),
        transforms.RandomApply([transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
        transforms.RandomGrayscale(p=0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.4914, 0.4822, 0.4465], 
                             [0.2023, 0.1994, 0.2010])
    ])
    return TwoViewTransform(transform)

def get_eval_transform():
    return transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize([0.4914, 0.4822, 0.4465], 
                             [0.2023, 0.1994, 0.2010])
    ])

# Loaders for the datasets
simclr_dataset = datasets.CIFAR10(root='./data', train=True, 
                                   download=True, transform=get_simclr_transform())
pool_dataset   = datasets.CIFAR10(root='./data', train=True, 
                                   download=False, transform=get_eval_transform())
test_dataset   = datasets.CIFAR10(root='./data', train=False, 
                                   download=False, transform=get_eval_transform())

simclr_loader = DataLoader(simclr_dataset, batch_size=512, shuffle=True, 
                            num_workers=4, pin_memory=True, drop_last=True)
pool_loader   = DataLoader(pool_dataset, batch_size=512, shuffle=False, 
                            num_workers=4, pin_memory=True)
test_loader   = DataLoader(test_dataset, batch_size=512, shuffle=False, 
                            num_workers=4, pin_memory=True)

print("Data loaded!")

Data loaded!


## Step 2: SimCLR Model Architecture

We implement the SimCLR model following the paper's specification (Appendix F.1):
- **Backbone**: ResNet18 encoder that extracts a 512-dimensional feature vector from each image
- **Projection head**: A 2-layer MLP that maps the 512-dim features down to 128 dimensions
- The contrastive loss is computed in the 128-dim projection space
- After training, the projection head is discarded, only the 512-dim encoder features are kept

In [4]:
class SimCLR(nn.Module):
    def __init__(self, projection_dim = 128):
        super().__init__()

        resnet = models.resnet18(weights=None)
        self.encoder = nn.Sequential(*list(resnet.children())[:-1])

        # head to downsize to 128 dimensions for upstream task
        self.projector = nn.Sequential(
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, projection_dim)
        )

    def forward(self,x):
        h = self.encoder(x).squeeze()
        z = self.projector(h)
        return h, z
    
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SimCLR().to(device)
print(f"Model on: {device}")

Model on: cuda


## Step 3: NT-Xent Contrastive Loss

The NT-Xent (Normalized Temperature-scaled Cross Entropy) loss is the training objective for SimCLR:
- For each image in the batch, two augmented views are created
- The loss pushes the two views of the **same** image close together in embedding space
- All other pairs in the batch are pushed apart
- Temperature parameter is set to 0.5, controlling how sharply pairs are separated

In [5]:
def nt_xent_loss(z1, z2, temperature=0.5):

    # normalizing the vectors
    z1 = nn.functional.normalize(z1, dim=1)
    z2 = nn.functional.normalize(z2, dim=1)

    # concatenate both views
    z = torch.cat([z1, z2], dim=0)

    sim = torch.mm(z, z.T) / temperature

    
    batch_size = z1.shape[0]
    mask = torch.eye(2 * batch_size, device=z.device).bool()
    sim.masked_fill_(mask, float('-inf'))
    
    labels = torch.cat([
        torch.arange(batch_size, 2 * batch_size),
        torch.arange(0, batch_size)
    ]).to(device)

    loss = nn.functional.cross_entropy(sim, labels)
    return loss

## Step 4: SimCLR Training

We train SimCLR for 500 epochs on the full CIFAR-10 training set (50,000 images) following Appendix F.1:
- **Optimiser**: SGD with momentum=0.9, weight decay=0.0001
- **Learning rate**: 0.4 with cosine annealing scheduler
- **Batch size**: 512
- Checkpoints are saved every 20 epochs in case of interruption
- No labels are used at any point during this training

In [5]:
def train_simclr(model, loader, epochs=500):
    optimizer = optim.SGD(model.parameters(), lr=0.4, 
                          momentum=0.9, weight_decay=0.0001)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for (view1, view2), _ in tqdm(loader, desc=f"Epoch {epoch+1}/{epochs}"):
            view1, view2 = view1.to(device), view2.to(device)
            
            _, z1 = model(view1)
            _, z2 = model(view2)
            
            loss = nt_xent_loss(z1, z2)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        scheduler.step()
        
        if (epoch + 1) % 20 == 0:
            avg_loss = total_loss / len(loader)
            print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}")
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'loss': avg_loss
            }, f'simclr_checkpoint_epoch{epoch+1}.pth') # I added this as the machine used for training 
            # occasionally switches off when faced with load so this is to resume training in case it does switch off.
    
    # Save the trained model
    torch.save(model.state_dict(), 'simclr_model.pth')
    print("Model saved!")

train_simclr(model, simclr_loader)

Epoch 20/500: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 97/97 [00:06<00:00, 15.03it/s]


Epoch 20, Loss: 5.7348


Epoch 40/500: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 97/97 [00:06<00:00, 14.50it/s]


Epoch 40, Loss: 5.6669


Epoch 60/500: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 97/97 [00:06<00:00, 15.40it/s]


...

Epoch 460, Loss: 5.4261


Epoch 480/500: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 97/97 [00:06<00:00, 16.05it/s]


Epoch 480, Loss: 5.4176


Epoch 500/500: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 97/97 [00:06<00:00, 15.22it/s]


Epoch 500, Loss: 5.4122
Model saved!


## Step 5: Extract and Save Embeddings

Once SimCLR is trained, we extract embeddings for every image in the training pool:
- We use the **encoder only** (512-dim), discarding the projection head
- Embeddings are **L2 normalised** as specified in the paper
- The order is kept consistent (shuffle=False) so indices match correctly
- Embeddings and labels are saved to disk for use in notebook 2

In [6]:
def extract_embeddings(model, loader, device):
    model.eval()
    embeddings = []
    labels = []
    
    with torch.no_grad():  # no gradients needed, saves memory
        for imgs, targets in tqdm(loader, desc="Extracting embeddings"):
            imgs = imgs.to(device)
            h, _ = model(imgs)  # take h (512-dim), discard z (128-dim)
            # L2 normalise as per paper
            h = nn.functional.normalize(h, dim=1)
            embeddings.append(h.cpu().numpy())
            labels.append(targets.numpy())
    
    embeddings = np.concatenate(embeddings, axis=0)  # shape [50000, 512]
    labels = np.concatenate(labels, axis=0)           # shape [50000]
    
    np.save('embeddings.pth', embeddings)
    np.save('labels.pth', labels)
    print(f"Embeddings shape: {embeddings.shape}")
    return embeddings, labels

embeddings, labels = extract_embeddings(model, pool_loader, device)

Extracting embeddings: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 98/98 [00:01<00:00, 87.56it/s]

Embeddings shape: (50000, 512)
